# 第1日 ノートブック01：AIとは何か

# Day 1 Notebook 01: Introduction to AI

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. コンピュータプログラムとは何かを説明できる | 1. Explain what a computer program is |
| 2. 自動化と AI の違いを説明できる | 2. Distinguish automation from artificial intelligence |
| 3. AI・機械学習・深層学習の関係を説明できる | 3. Describe how AI, machine learning and deep learning relate |
| 4. 「学習する」と「推論する」の違いを説明できる | 4. Explain the difference between training and inference |
| 5. AI が間違える理由を、製造の例で説明できる | 5. Explain why AI makes mistakes, using manufacturing examples |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 115 分 / 115 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | `00_Environment_Setup.ipynb` を完了していること。 / You have completed `00_Environment_Setup.ipynb`. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第1日 / Day 1](../../COURSE_SYLLABUS.md#day1) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| このノートブックの実行結果 | Execution results of this notebook |
| 演習 1〜3 の回答 | Answers to exercises 1-3 |
| 理解度チェックの回答 | Answers to the understanding questions |

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第1日 / Day 1 |
| この日の順番 / Position today | 1 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 115 分 / 115 minutes |
| 前提 / Prerequisites | `00_Environment_Setup.ipynb` |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第1日 / Day 1](../../COURSE_SYLLABUS.md#day1) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


---

## Part 0　準備 / Preparation

#### このノートブックの環境を確認する
まず研修フォルダを見つけ、必要なライブラリを確認します。エラーが出た場合は `docs/troubleshooting/installation.md` を見てください。
#### Check the environment for this notebook
First locate the course folder and check the libraries. If something is missing, see `docs/troubleshooting/installation.md`.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

#### 図の日本語フォントを設定する
Matplotlib の既定フォントは日本語を表示できません。そのままだと図のタイトルが `□□□` になります。次のセルで、日本語を表示できるフォントを探して設定します。Mac や Linux では自動で別のフォントが選ばれます。
#### Configure Japanese text in figures
Matplotlib's default font cannot draw Japanese characters, so figure titles would appear as `□□□`. The next cell looks for a Japanese font and selects it. On macOS or Linux a different font will be chosen automatically.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def show_png(relative_parts, caption=""):
    """素材フォルダの PNG を表示する / display a PNG from the assets folder."""
    path = ASSETS.joinpath(*relative_parts)
    if not path.is_file():
        print("画像がありません / file not found:", path)
        print("assets\\scripts\\generate_training_assets.py を実行してください。")
        return
    image = Image.open(path)
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.axis("off")
    if caption:
        plt.title(caption)
    plt.show()
    print("size (width, height) =", image.size, " mode =", image.mode)

---

## Part 1　コンピュータプログラムとは / What is a computer program

#### プログラムは「手順書」です
プログラムとは、コンピュータに対する **手順書** です。
「この順番で、これを計算しなさい」と細かく書いたものです。

コンピュータは賢くありません。書かれたとおりに、正確に、速く実行するだけです。
その代わり、**疲れない・迷わない・同じ結果を出す** という大きな長所があります。

製造の例で考えてみましょう。シートの幅を測って、規格内かどうかを判定する仕事です。

1. 測定値を読む
2. 規格値 100.0 mm との差を計算する
3. 差が ±0.5 mm 以内なら「OK」、そうでなければ「NG」
4. 結果を記録する

この 4 ステップは、**人間が決めたルール** です。プログラムはこれをそのまま実行します。
#### A program is a set of instructions
A program is a **set of instructions** for a computer: step by step, precisely.

A computer is not clever. It executes exactly what is written — no more, no less. Its advantage is
that it never gets tired, never hesitates, and always produces the same result for the same input.

A manufacturing example: measuring sheet width and judging whether it is within specification.

1. read the measured value
2. compute the difference from the nominal 100.0 mm
3. if the difference is within ±0.5 mm, the result is "OK", otherwise "NG"
4. record the result

These four steps are **rules written by a human**. The program simply follows them.

#### 実際に書いてみる
下のセルは、上の 4 ステップをそのまま Python にしたものです。
`def` は「関数（処理のまとまり）を定義する」という意味です。今は
「`judge_width` という名前で、この 4 ステップをまとめた」と理解してください。
#### Let us write it
The cell below is exactly those four steps written in Python.
`def` defines a **function** — a named block of steps. For now, read it as
"the name `judge_width` refers to these four steps".

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
def judge_width(measured_mm, nominal_mm=100.0, tolerance_mm=0.5):
    """幅を判定する関数 / judge a measured width."""
    difference = measured_mm - nominal_mm          # 1. 差を計算 / compute the difference
    if abs(difference) <= tolerance_mm:            # 2. 規格内か / inside tolerance?
        return "OK"
    return "NG"                                    # 3. 規格外 / outside tolerance


for value in [100.0, 100.3, 99.7, 100.6, 99.4, 103.2]:
    print(f"測定値 {value:6.1f} mm -> {judge_width(value)}")

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
測定値  100.0 mm -> OK
測定値  100.3 mm -> OK
測定値   99.7 mm -> OK
測定値  100.6 mm -> NG
測定値   99.4 mm -> NG
測定値  103.2 mm -> NG
```

You should see something like the following (exact numbers vary slightly between machines).

```text
measured  100.0 mm -> OK
measured  100.3 mm -> OK
measured   99.7 mm -> OK
measured  100.6 mm -> NG
measured   99.4 mm -> NG
measured  103.2 mm -> NG
```

#### このプログラムの限界
このプログラムは、**幅** については完璧に動きます。速くて、疲れません。

しかし、次の仕事を頼むと困ります。

| 仕事 | 人手で書けるルールは？ |
|---|---|
| シート表面の「黒い点」を見つける | 「黒い」とは数値でいくつ？ 光の当たり方で変わるのでは？ |
| 傷と、単なる模様の違いを見分ける | ルールを言葉で書ききれない |
| 「いつもと違う」を見つける | 「いつも」をどう定義する？ |

ここが AI の出番です。ルールを人間が書く代わりに、
**データからルール（のようなもの）をコンピュータ自身に作らせます。**

#### Where this program stops
For width this program is perfect: fast, tireless, exact.

But try to give it the following tasks:

| Task | Can you write the rule by hand? |
|---|---|
| find "dark spots" on the sheet surface | how dark is dark? it depends on the lighting |
| tell a scratch from a harmless pattern | impossible to express in words |
| find "something different from usual" | how do you define "usual"? |

This is where AI starts: instead of a human writing the rule, the computer builds the rule
(or something rule-like) from **data**.

---

## Part 2　自動化とは / What is automation

#### 自動化＝同じ手順を機械に繰り返させる
自動化とは、**決まった手順を、機械に繰り返させる** ことです。
判断のルールは人間が決めます。機械は繰り返すだけです。

製造現場での例：

- コンベアで製品を運ぶ（自動化）
- 一定のタイミングでカメラのシャッターを切る（自動化）
- 測定値が規格内か判定する（自動化。ルールは上で書いたとおり）

自動化は、ルールが **はっきり書ける** 仕事に非常に強いです。

#### Automation = letting a machine repeat a fixed procedure
Automation means making a machine **repeat a fixed procedure**. The decision rule is
written by a human; the machine only repeats it.

Examples on a production line: moving products on a conveyor, triggering the camera at a fixed
timing, judging a measured value against a specification.

Automation is extremely powerful whenever the rule **can be written down clearly**.

### 検査システムの 4 つの役割 / the four roles in an inspection system

![](../../assets/diagrams/teach_inspection_roles.png)

図：検査を自動化するときの「撮る・測る・決める・動かす」の 4 つの役割と、AI が受け持つ範囲（模式図）。

Figure: the four roles of an automated inspection — capture, measure, decide, act — and the part AI covers (schematic).

**気づいてほしいこと**

AI が受け持つのは ③「決める」だけです。①撮る・②測る・④動かす が決まっていないと、③ を良くしても検査全体は良くなりません。数で書ける判定は if 文のままにします。

**What to notice**

AI only covers ③ decide. If capture, measure and act are not fixed, a better ③ does not make a better inspection. Decisions you can write as numbers stay as if-statements.

**考えてみてください / Try to answer**

- この 4 つのうち、いま自分の工程で決まっていないのはどれですか。
- Which of the four is not yet decided in your own line today?


#### 繰り返しを実際に見る
1 日に 10,000 枚を検査するとします。上の関数を 10,000 回呼べば、
人間なら数時間かかる作業が 1 秒で終わります。

下のセルは、1,000 個の測定値を自動で判定し、NG の数を数えます。
`for` は「繰り返す」という意味です。
#### See the repetition
Imagine inspecting 10,000 sheets per day. Calling that function 10,000 times finishes
in under a second, while a human would need hours.

The next cell judges 1,000 measured values automatically and counts the NG results.
`for` means "repeat".

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import numpy as np

rng = np.random.default_rng(seed=1)
# 平均 100.05 mm、ばらつき 0.3 mm の測定値を 1000 個作る
# 1000 measurements with mean 100.05 mm and standard deviation 0.3 mm
measurements = rng.normal(loc=100.05, scale=0.3, size=1000)

results = [judge_width(float(value)) for value in measurements]
ng_count = results.count("NG")

print("検査した枚数 / inspected :", len(results))
print("OK の枚数   / OK count   :", results.count("OK"))
print("NG の枚数   / NG count   :", ng_count)
print(f"NG 率       / NG rate    : {ng_count / len(results) * 100:.2f} %")

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
検査した枚数 / inspected : 1000
OK の枚数   / OK count   : 910
NG の枚数   / NG count   : 90
NG 率       / NG rate    : 9.00 %
```

You should see something like the following (exact numbers vary slightly between machines).

```text
inspected : 1000
OK count  : 910
NG count  : 90
NG rate   : 9.00 %
```

#### 自動化で足りないとき
この判定は、**幅という 1 つの数値** だからうまくいきました。
ところが現場では、こんな依頼が来ます。

> 「シートの表面に、ときどき小さな黒い点が付く。これを自動で見つけたい。」

「黒い点」のルールを、あなたは数値で書けますか？

- 明るさが 100 未満 → 照明が暗い日は全部 NG になる
- 直径 3 mm 以上 → 1 mm の点は見逃す
- 形が丸い → 傷は見逃す

ルールの組み合わせは際限なく増え、しかも **現場ごとに違います**。
ここで AI が必要になります。

#### When automation is not enough
This worked because width is **a single number**. But on the shop floor the request is
different:

> "Small dark spots sometimes appear on the sheet surface. We want to find them automatically."

Can you write the rule for "dark spot" as numbers?

- brightness below 100 -> every darker lighting day becomes NG
- diameter above 3 mm -> 1 mm spots are missed
- round shape -> scratches are missed

The combinations never end, and they differ per line. That is why AI becomes necessary.

---

## Part 3　AI・機械学習・深層学習 / AI, machine learning, deep learning

#### 3 つの言葉の関係
この 3 つは、別々の技術ではなく **入れ子（大小関係）** になっています。

```text
人工知能 AI
 └── 機械学習 Machine Learning
      └── 深層学習 Deep Learning
```

| 言葉 | 意味（やさしく） | 製造での例 |
|---|---|---|
| 人工知能 AI | 人間の知的な仕事をコンピュータで代替する考え方の総称 | 「人の目でやっていた検査を機械でやる」 |
| 機械学習 ML | データから規則性を見つける仕組み | 正常品 500 枚の画像から「正常の範囲」を学ぶ |
| 深層学習 DL | 多層のニューラルネットワークを使う機械学習 | 画像から欠陥を見つける（YOLO、PatchCore など） |

つまり、深層学習は機械学習の一種で、機械学習は AI の一種です。
「AI」と言ったとき、それが必ず深層学習とは限りません。

#### How the three terms relate
The three terms are **nested**, not separate technologies:

```text
Artificial Intelligence (AI)
 └── Machine Learning (ML)
      └── Deep Learning (DL)
```

| Term | Plain meaning | Manufacturing example |
|---|---|---|
| AI | the general idea of doing human-like work with computers | "replace visual inspection with a machine" |
| ML | finding regularities in data | learn "the range of normal" from 500 good images |
| DL | ML with many-layered neural networks | finding defects in images (YOLO, PatchCore) |

So deep learning is a kind of machine learning, and machine learning is a kind of AI.
Saying "AI" does not automatically mean deep learning.

### AI・機械学習・深層学習の入れ子 / how the three words nest

![](../../assets/diagrams/teach_ai_ml_dl_nested.png)

図：AI ⊃ 機械学習 ⊃ 深層学習 の包含関係と、それぞれの例（模式図）。

Figure: AI ⊃ machine learning ⊃ deep learning, with an example of each (schematic).

**気づいてほしいこと**

**すべての AI が機械学習ではありません。** 人が書いた if 文も AI の一部ですが、機械学習ではありません。この研修で使う YOLO と PatchCore は、いちばん内側の深層学習です。

**What to notice**

**Not all AI is machine learning.** An if-statement a person wrote can count as AI but is not machine learning. YOLO and PatchCore sit in the innermost ring.

**考えてみてください / Try to answer**

- 「幅が 99.5 mm 未満なら NG」という if 文は、3 つのどれに入りますか。
- Where does the rule “NG if width < 99.5 mm” belong: AI, machine learning or deep learning?


### 学習と推論は別の作業 / training and inference are different jobs

![](../../assets/diagrams/teach_train_vs_inference.png)

図：学習（training）と推論（inference）の違い。学習は 1 回だけで時間がかかり、推論は部品ごとに何度でも行えて速い、という役割の違いです（模式図）。

Figure: training versus inference. Training happens once and is slow; inference runs on every part and is fast (schematic).

**気づいてほしいこと**

重み（weights）は学習で決まり、推論では**変わりません**。現場で毎日動くのは推論のほうです。モデルを良くしたいときは、推論ではなく学習をやり直します。

**What to notice**

The weights are fixed by training and **do not change** during inference. What runs in the factory is inference. To make the model better you retrain; you do not tune the inference.

**考えてみてください / Try to answer**

- 1 日に 1000 個の部品を検査するとしたら、学習と推論はそれぞれ何回行われますか。
- For 1000 parts a day, how many training and how many inference runs happen?


---

## Part 4　ルールと学習の違い / Fixed rules versus learned patterns

### 決まりを書く検査と、例から学ぶ検査 / rule-based versus learned inspection

![](../../assets/diagrams/teach_rule_vs_learned.png)

図：上がルールで書ける検査、下が例から学ぶ検査。

Figure: the top row is an inspection a rule can describe; the bottom row learns from examples.

**気づいてほしいこと**

上の段では、人が「99.5 mm 未満は NG」と決めます。下の段では、人は決まりを書きません。かわりに正常品と異常品の例を渡し、コンピュータが特徴を学びます。

**What to notice**

In the top row a person writes the rule ("below 99.5 mm is NG"). In the bottom row nobody writes a rule: examples of good and bad parts are supplied and the computer learns the features.

**考えてみてください / Try to answer**

- あなたの工程で、いま「決まり」として書ける検査はどれですか。
- Which inspections on your line can be written as a rule today?


#### 同じ仕事を、2 つのやり方で解く
ここでは「表面の明るさの平均値」から OK/NG を判定する仕事を、
2 通りで解きます。

**やり方 A（ルール）**：人間がしきい値を決める。例「平均 200 未満なら NG」。

**やり方 B（学習）**：正常品のデータだけを見て、しきい値をコンピュータに決めさせる。

やり方 B は、実は **一番単純な機械学習** です。データの平均とばらつきから
「正常の範囲」を計算しているだけですが、「データから決めた」という点が決定的に違います。

#### Solve the same task in two ways
Let us solve one task — judging OK/NG from the average brightness of the surface —
in two ways.

**Way A (rule):** a human picks the threshold, e.g. "average below 200 means NG".

**Way B (learning):** show the computer only good products, and let it determine the threshold.

Way B is **real machine learning in its simplest form**: it computes "the normal range" from the
mean and spread of the data. The crucial difference is that the threshold came **from data**.

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import numpy as np

rng = np.random.default_rng(seed=42)

# 正常品 200 枚の「明るさの平均値」。平均 205、ばらつき 4。
# 200 good products: mean brightness around 205 with a spread of about 4.
good_samples = rng.normal(loc=205.0, scale=4.0, size=200)

print("正常品のデータ / good samples")
print("  平均 mean :", round(float(good_samples.mean()), 2))
print("  標準偏差 std:", round(float(good_samples.std(ddof=1)), 2))
print("  最小 min  :", round(float(good_samples.min()), 2))
print("  最大 max  :", round(float(good_samples.max()), 2))

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
正常品のデータ / good samples
  平均 mean : 204.88
  標準偏差 std: 3.53
  最小 min  : 196.47
  最大 max  : 216.66
```

You should see something like the following (exact numbers vary slightly between machines).

```text
good samples
  mean : 204.88
  std  : 3.53
  min  : 196.47
  max  : 216.66
```

### ±3σ が使えるのは、どんな分布か / when the ±3σ rule applies

![](../../assets/diagrams/teach_gaussian_vs_skewed.png)

図：左右対称の分布と、右に裾を引く分布に、同じ「平均 ± 3σ」を当てはめた結果（合成データ。入る割合は計算値）。

Figure: the same “mean ± 3σ” rule applied to a symmetric and to a right-skewed distribution (synthetic data; the coverage is computed).

**気づいてほしいこと**

99.7 % は**正規分布（左右対称）のときの性質**です。右のように裾を引く分布では、同じ ±3σ でも入る割合が変わります。「99.7 %」と書く前に、ヒストグラムの形を確かめてください。

**What to notice**

99.7 % is a property of the **normal (symmetric) distribution**. For a skewed one the same ±3σ rule covers a different share. Look at the histogram before quoting 99.7 %.

**考えてみてください / Try to answer**

- この右の分布で「平均 ± 3σ の外は異常」と決めると、どんな見逃し・誤報が起きますか。
- If you call everything outside mean ± 3σ abnormal on the right-hand distribution, what misses and false alarms follow?


#### しきい値をデータから決める
正常品の値が **左右対称（正規分布）にばらつく** とき、その 99.7 % は
「平均 ± 3 × 標準偏差」に入ります。**この 99.7 % は、正規分布という仮定の上での性質です。**
分布が左右どちらかに裾を引いていると、同じ ±3σ でも入る割合は変わります（上の図を見てください）。

この研修のデータは正規分布に近いので、ここでは「平均 ± 3σ」を正常の範囲とします。
これが **学習** の結果です。

```text
下限 = 平均 - 3 × 標準偏差
上限 = 平均 + 3 × 標準偏差
```

#### Determine the threshold from data
When good parts scatter **symmetrically (a normal distribution)**, 99.7 % of them fall inside
"mean ± 3 × standard deviation". **That 99.7 % is a property of the normal-distribution
assumption**: for a skewed distribution the same ±3σ covers a different share (see the figure
above). This course's data is close to symmetric, so we take mean ± 3σ as the normal range.
That is the **result of training**.

```text
lower = mean - 3 * std
upper = mean + 3 * std
```

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# --- 学習 / training: 正常品データだけから範囲を決める ---
mean_brightness = float(good_samples.mean())
std_brightness = float(good_samples.std(ddof=1))

lower_limit = mean_brightness - 3 * std_brightness
upper_limit = mean_brightness + 3 * std_brightness

print(f"学習した正常範囲 / learned normal range: {lower_limit:.1f} 〜 {upper_limit:.1f}")


def judge_brightness(value, low, high):
    """学習した範囲で判定する / judge using the learned range."""
    return "OK" if low <= value <= high else "NG"


# --- 推論 / inference: 新しい測定値に当てはめる ---
new_samples = [
    ("正常に近い / near normal", 206.0),
    ("少し暗い   / slightly dark", 199.0),
    ("かなり暗い / clearly dark ", 188.0),
    ("明るすぎ   / too bright   ", 218.0),
]

print()
for label, value in new_samples:
    print(f"{label}  brightness={value:6.1f}  ->  {judge_brightness(value, lower_limit, upper_limit)}")

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
学習した正常範囲 / learned normal range: 194.3 〜 215.5

正常に近い / near normal  brightness= 206.0  ->  OK
少し暗い   / slightly dark  brightness= 199.0  ->  OK
かなり暗い / clearly dark   brightness= 188.0  ->  NG
明るすぎ   / too bright     brightness= 218.0  ->  NG
```

You should see something like the following (exact numbers vary slightly between machines).

```text
learned normal range: 194.3 - 215.5

near normal   brightness= 206.0  ->  OK
slightly dark brightness= 199.0  ->  OK
clearly dark  brightness= 188.0  ->  NG
too bright    brightness= 218.0  ->  NG
```

#### 「学習した」とはどういうことか
このプログラムは、`mean_brightness` と `std_brightness` という
**2 つの数値** をデータから計算しました。この 2 つの数値が「学習した結果」です。

本格的な深層学習では、この数値が数十万〜数億個になります。しかし **考え方は同じ** です。

- **学習（training）**：データを見て、内部の数値（パラメータ）を決める
- **推論（inference）**：決まった数値を使って、新しいデータを判定する

> 大事な区別です。学習は **時間がかかり、データが必要** です。
> 推論は **速く、データは 1 枚で足ります**。工場のラインで毎秒動くのは推論のほうです。

#### What 'trained' actually means
The program computed **two numbers** from data: `mean_brightness` and
`std_brightness`. Those two numbers are the training result.

In real deep learning those numbers become hundreds of thousands to hundreds of millions.
The idea is identical:

- **Training**: look at data and determine the internal numbers (parameters)
- **Inference**: use those fixed numbers to judge new data

This distinction matters: training is slow and needs data; inference is fast and needs one image.
The part that runs every second on a production line is inference.

---

## Part 5　学習データと、AI が間違える理由 / Training data and why AI is wrong

### 照明が変わると、正常品が NG になる / changing the light makes good parts fail

![](../../assets/diagrams/teach_brightness_shift.png)

図：正常品の明るさの分布（青）と、照明が変わった日の分布（黄）。赤線は学習した下限。

Figure: brightness of good parts (blue) and of good parts on a day with different lighting (yellow). The red line is the learned lower limit.

**気づいてほしいこと**

学習したのは「青い分布」だけです。照明が暗くなると、同じ正常品が黄色い分布に移り、赤線より下になって NG と判定されます。これが **偽陽性（false reject）** です。モデルは壊れていません。**学習したときと違う条件** が原因です。

「条件」とは、この例では照明の明るさです。現場では次のどれが変わっても同じことが起きます。

| 変わると効くもの / what can change | 例 / example |
|---|---|
| 照明の明るさ・色温度 / light level and colour temperature | 蛍光灯を LED に替えた、夕方になった |
| カメラの露出・ゲイン / exposure and gain | オート露出が効いた |
| レンズ・距離・角度 / lens, distance, angle | 治具を交換した、レンズが汚れた |
| 背景・周囲の反射 / background and reflections | 棚を移動した、カバーを外した |

だから記録するのは「スコア」だけでなく、**その日、何を、どの条件で撮ったか** です。

「条件」とは、この例では照明の明るさです。現場では次のどれが変わっても同じことが起きます。

| 変わると効くもの / what can change | 例 / example |
|---|---|
| 照明の明るさ・色温度 / light level and colour temperature | 蛍光灯を LED に替えた、夕方になった |
| カメラの露出・ゲイン / exposure and gain | オート露出が効いた |
| レンズ・距離・角度 / lens, distance, angle | 治具を交換した、レンズが汚れた |
| 背景・周囲の反射 / background and reflections | 棚を移動した、カバーを外した |

だから記録するのは「スコア」だけでなく、**その日、何を、どの条件で撮ったか** です。

「条件」とは、この例では照明の明るさです。現場では次のどれが変わっても同じことが起きます。

| 変わると効くもの / what can change | 例 / example |
|---|---|
| 照明の明るさ・色温度 / light level and colour temperature | 蛍光灯を LED に替えた、夕方になった |
| カメラの露出・ゲイン / exposure and gain | オート露出が効いた |
| レンズ・距離・角度 / lens, distance, angle | 治具を交換した、レンズが汚れた |
| 背景・周囲の反射 / background and reflections | 棚を移動した、カバーを外した |

だから記録するのは「スコア」だけでなく、**その日、何を、どの条件で撮ったか** です。

**What to notice**

Only the blue distribution was used for training. With dimmer light the same good parts move into the yellow distribution, fall below the red line and are flagged NG: a **false reject**. The model is not broken; the **conditions differ from training**.

Here the condition is brightness. On a line, any of these changes the same way:

| what can change | example |
|---|---|
| light level and colour temperature | fluorescent replaced by LED, late afternoon |
| exposure and gain | auto-exposure kicks in |
| lens, distance, angle | a new fixture, a dirty lens |
| background and reflections | a shelf moved, a cover removed |

So record **what you photographed, on which day, under which conditions** — not only the score.

Here the condition is brightness. On a line, any of these changes the same way:

| what can change | example |
|---|---|
| light level and colour temperature | fluorescent replaced by LED, late afternoon |
| exposure and gain | auto-exposure kicks in |
| lens, distance, angle | a new fixture, a dirty lens |
| background and reflections | a shelf moved, a cover removed |

So record **what you photographed, on which day, under which conditions** — not only the score.

Here the condition is brightness. On a line, any of these changes the same way:

| what can change | example |
|---|---|
| light level and colour temperature | fluorescent replaced by LED, late afternoon |
| exposure and gain | auto-exposure kicks in |
| lens, distance, angle | a new fixture, a dirty lens |
| background and reflections | a shelf moved, a cover removed |

So record **what you photographed, on which day, under which conditions** — not only the score.

**考えてみてください / Try to answer**

- この図と同じことが起きないように、現場では何を記録しておくべきですか。
- What would you record on the line so that this cannot happen unnoticed?


#### 学習データが偏ると、AI も偏る
上の学習は「正常品 200 枚」だけを使いました。もしその 200 枚が
**蛍光灯が暗い日に撮った写真だけ** だったら、どうなるでしょうか。

学習した正常範囲は「暗い側」にずれます。すると、明るい日に撮った **正常品** を
NG と判定してしまいます。これが **偽陽性（false positive）** です。

逆に、正常範囲が広すぎると、**本当の欠陥** を見逃します。これが
**偽陰性（false negative）** です。

製造の検査では、**偽陰性（見逃し）のほうが深刻** です。不良品を市場に出すことになるからです。

#### Biased training data makes a biased model
The training above used 200 good products. What if all 200 photos were taken on a
dark, cloudy day? The learned normal range shifts toward the dark side, and a good product
photographed under bright light is judged NG. That is a **false positive**.

If the range is too wide, real defects pass unnoticed. That is a **false negative**.

In manufacturing inspection, **false negatives are the serious ones** — a defective product reaches
the customer.

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# 暗い日に撮った写真だけで学習してしまった場合 / trained only on a dark day
dark_day_samples = rng.normal(loc=194.0, scale=3.0, size=200)

mean_dark = float(dark_day_samples.mean())
std_dark = float(dark_day_samples.std(ddof=1))
low_dark = mean_dark - 3 * std_dark
high_dark = mean_dark + 3 * std_dark

print(f"暗い日のデータで学習した範囲 / range learned from a dark day: {low_dark:.1f} 〜 {high_dark:.1f}")
print()

true_good_product = 206.0   # 明るい日に撮った、まったく正常な製品 / a perfectly good product

print("本来は OK の製品 / this product is actually OK :", true_good_product)
print("このモデルの判定 / this model says          :", judge_brightness(true_good_product, low_dark, high_dark))
print()
print("=> 偽陽性（良品を NG と判定）/ false positive: good product flagged as NG")

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
暗い日のデータで学習した範囲 / range learned from a dark day: 184.9 〜 203.2

本来は OK の製品 / this product is actually OK : 206.0
このモデルの判定 / this model says          : NG

=> 偽陽性（良品を NG と判定）/ false positive: good product flagged as NG
```

You should see something like the following (exact numbers vary slightly between machines).

```text
range learned from a dark day: 184.9 - 203.2

this product is actually OK : 206.0
this model says             : NG

=> false positive: good product flagged as NG
```

#### AI が間違える理由（まとめ）
原因は、大きく 5 つです。

1. **学習データが足りない** — 正常品 5 枚では「正常」を学べない
2. **学習データが偏っている** — 暗い日だけ、特定の品種だけ、など
3. **学習データに答え（ラベル）の間違いがある** — 欠陥を見落として「正常」とラベル付け
4. **本番と条件が違う** — 照明・レンズ・距離・速度が変わった
5. **そもそも見えない** — 0.3 mm の欠陥が、解像度不足で写っていない

5 番目は、AI ではなく **カメラと光学** の問題です。第 2 週で扱います。

#### Why AI makes mistakes (summary)
Five main causes:

1. **not enough training data** — five good images cannot describe "normal"
2. **biased training data** — only dark days, only one product type
3. **wrong labels** — a missed defect labelled "normal"
4. **different conditions in production** — lighting, lens, distance or speed changed
5. **the defect is not visible at all** — a 0.3 mm defect lost to insufficient resolution

Cause 5 is not an AI problem but a **camera and optics** problem. Week 2 covers it.

---

## Part 6　誘導実験 / Guided experiment

#### パラメータを変えて、結果の変化を予想する
下のセルには 3 つのつまみがあります。**実行する前に**、結果を予想してください。

| つまみ | 変えてみる値 | 予想すること |
|---|---|---|
| `k`（何×標準偏差か） | 1, 2, 3, 4 | NG の数は増える？ 減る？ |
| `good_mean`（正常品の平均） | 190, 205, 215 | 学習結果は？ |
| `n_good`（学習枚数） | 10, 200, 2000 | 学習した範囲は安定する？ |

#### Change parameters and predict what happens
The cell below has three knobs. **Predict the outcome before you run it.**

| Knob | Try | Predict |
|---|---|---|
| `k` (how many standard deviations) | 1, 2, 3, 4 | does the NG count go up or down? |
| `good_mean` (average of good products) | 190, 205, 215 | what is learned? |
| `n_good` (number of training images) | 10, 200, 2000 | is the learned range stable? |

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# ===== ここを書き換えてください / edit these =====
k = 3
good_mean = 205.0
good_std = 4.0
n_good = 200
n_test = 500
# =================================================

rng2 = np.random.default_rng(seed=7)
train = rng2.normal(loc=good_mean, scale=good_std, size=n_good)
test = rng2.normal(loc=good_mean, scale=good_std, size=n_test)

low = train.mean() - k * train.std(ddof=1)
high = train.mean() + k * train.std(ddof=1)

flagged = sum(1 for v in test if not (low <= v <= high))

print(f"学習に使った枚数 / training images : {n_good}")
print(f"学習した正常範囲 / learned range   : {low:.2f} 〜 {high:.2f}")
print(f"検査した枚数     / tested          : {n_test}")
print(f"NG と判定した数  / flagged as NG   : {flagged}  ({flagged / n_test * 100:.2f} %)")
print()
print("すべて正常品なので、この NG はすべて偽陽性です。")
print("All products here are good, so every NG is a false positive.")

#### 気づいてほしいこと
`n_good` を 10 にすると、結果が実行のたびに大きく変わります。
`n_good` を 2000 にすると、安定します。

**学習データの枚数が少ないと、AI の判断は不安定になります。**
第 4 日・第 5 日で扱う本格的なモデルでも、同じことが起きます。

#### What you should notice
With `n_good = 10` the result changes a lot every run; with `n_good = 2000` it becomes
stable. **Too little training data makes the decision unstable** — true for the real models on
Day 4 and Day 5 as well.

---

## 演習 / Exercises

#### 演習 1（基本）：用意された関数を **動かして** 予想する
#### Exercise 1 (basic): run the supplied function and predict its result

下のセルには、**もう書いてある関数** `judge_thickness` があります。第2日で関数の書き方を学びます。
今日は **書き換えずに実行** し、結果を予想してください。

The cell below already contains `judge_thickness`. Functions are taught on Day 2; today you
**run it without editing** and predict the result.

| 予想 / Predict | 値 / Value |
|---|---|
| 規格内 / inside the tolerance | 1.95, 2.0, 2.1 |
| 規格外 / outside the tolerance | 2.15 |

**できたら:** 予想と結果が違った値を 1 つ書き、なぜ違ったかを一文で説明してください。
**Done when:** one value where your prediction was wrong is written down, with a one-sentence reason.

> 自分の関数を書く練習は、**第2日ノートブック02 の演習 1** にあります。
> Writing your own function is **Exercise 1 of Day 2 Notebook 02**.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# この関数は「読むための例」です。第2日で自分で書けるようになります。
# This function is a worked example to read. You will write your own on Day 2.
def judge_thickness(measured, nominal=2.0, tolerance=0.1):
    """厚さを判定する / judge a measured thickness."""
    if abs(measured - nominal) <= tolerance:
        return "OK"
    return "NG"


for value in [1.95, 2.0, 2.1, 2.15]:
    print(f"{value:>5.2f} mm -> {judge_thickness(value)}")

print()
print("予想と結果をくらべてください / compare your predictions with these results:")

#### 演習 2（中級）：正常範囲を広げる／狭める
Part 6 の `k` を 1 から 4 まで変えて、偽陽性の数を表にまとめてください。
「k を大きくすると、偽陽性は減るが、何が増えるか？」を 1 行で書いてください。

#### Exercise 2 (intermediate): widen or narrow the normal range
Vary `k` from 1 to 4 in Part 6, record the false-positive counts in a table, and
answer in one line: "increasing k reduces false positives, but what increases?"
Done when the table has four rows and your one-line answer is written as a comment.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
# 結果の表 / table of results
# k=1 : ____ NG
# k=2 : ____ NG
# k=3 : ____ NG
# k=4 : ____ NG

# k を大きくすると偽陽性は減るが、何が増えるか： ________________________________

#### 演習 3（発展）（発展・任意）：なぜ AI が必要かを説明する
「幅の判定」はルールで書けましたが、「黒点の検出」は書けませんでした。
理由を、あなたの現場の言葉で 3 行以内にまとめてください。そのうえで、
ルールで書ける部分と AI が必要な部分を、次の表に書き分けてください。

#### Exercise 3 (challenge) (challenge, optional) explain why AI is needed
"Width judgement" could be written as a rule; "spot detection" could not.
Explain why in three lines using your own shop-floor language, then fill in the table below
separating what a rule can do from what needs AI.
Done when the table lists at least three tasks on each side.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
# 理由 / reasoning:
# 1.
# 2.
# 3.

# ルールで書ける / rule is enough:     1.            2.            3.
# AI が必要 / AI is needed:             1.            2.            3.

---

#### 理解度チェック / Understanding questions
次の問いに、自分のことばで答えてください。

1. 「プログラム」と「AI」の違いを、あなたの職場の作業を例に説明してください。
2. 機械学習と深層学習は、どちらが広い概念ですか。理由も書いてください。
3. 「学習」と「推論」の違いを、時間とデータの量の観点から説明してください。
4. 正常品だけを撮った写真で学習したモデルが、実際のラインで誤判定する原因を 2 つ挙げてください。
5. 偽陽性と偽陰性のうち、製造検査でより深刻なのはどちらですか。理由も書いてください。

Answer in your own words.

1. Explain the difference between "a program" and "AI", using a task from your workplace.
2. Which is the broader idea: machine learning or deep learning? Explain why.
3. Explain training versus inference in terms of time and data.
4. Give two reasons why a model trained on good-product photos only could misjudge on the real line.
5. Which is more serious in manufacturing inspection: a false positive or a false negative? Why?
Model answers are in `docs/instructor/day01_instructor_guide.md`.
Write your own answer before reading them.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `ModuleNotFoundError: No module named 'numpy'` | `ModuleNotFoundError: No module named 'numpy'` | カーネルが違います。`sys.executable` を確認し、`Kernel` → `Change Kernel` → `Python 3 (AI Vision)`。 | Wrong kernel. Check `sys.executable`, then choose `Python 3 (AI Vision)`. |
| 数値が、このノートブックの例と少し違う | My numbers differ slightly from the examples | 正常です。乱数の出発点が同じでも、OS や NumPy の版で末尾の桁が変わります。 | Normal. The last digits vary between OS and NumPy versions. |
| `NameError: name '...' is not defined` | `NameError: name '...' is not defined` | 前のセルを実行していません。上から順に実行し直してください。 | An earlier cell was not run. Run the cells again from the top. |
| `TypeError: 'list' object is not callable` | `TypeError: 'list' object is not callable` | 変数名に `list` や `sum` など組み込みの名前を使うと起こります。別の名前にしてください。 | You used a built-in name such as `list` or `sum`. Rename the variable. |
| メモリ不足（`MemoryError`） | `MemoryError` | この演習では通常起きません。`n_good` に極端に大きな値を入れた場合は小さくしてください。 | Unusual here. If you set `n_good` extremely large, reduce it. |

## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 6 の説明を読んだ
      - [ ] Read Parts 1-6
- [ ] すべてのコードセルを上から順に実行した
      - [ ] Ran every code cell from top to bottom
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 誘導実験で `k` と `n_good` を変えてみた
      - [ ] Changed `k` and `n_good` in the guided experiment
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] 偽陽性と偽陰性を、自分のことばで説明できる
      - [ ] Can explain false positive / false negative in your own words
- [ ] エラーの記録を残した（無ければ「なし」）
      - [ ] Recorded errors (or wrote 'none')
- [ ] このノートブックを保存した
      - [ ] Saved this notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Python 公式チュートリアル（日本語）](https://docs.python.org/ja/3/tutorial/)
- [scikit-learn: 機械学習の考え方](https://scikit-learn.org/stable/getting_started.html)
- [NumPy 乱数生成器](https://numpy.org/doc/stable/reference/random/generator.html)
- [広島化成 参考資料](../../docs/reference_materials.md)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

`good_std` を 1.0 と 10.0 にして、学習した正常範囲の「幅」がどう変わるか調べてください。幅が広いほど、何を見逃しやすくなりますか？

Set `good_std` to 1.0 and to 10.0 and compare the width of the learned normal range. The wider the range, the more likely you are to miss what?